# Train Faster R-CNN, EfficientDet, RT-DETR

Notebook này dùng dataset YOLO có `data.yaml` giống notebook `shrimp.ipynb` gốc. Faster R-CNN và EfficientDet dùng PyTorch dataset đọc trực tiếp label YOLO; RT-DETR dùng Ultralytics train trực tiếp từ `data.yaml`.


## Mount Google Drive


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


## Install Dependencies


In [ ]:

!pip install -q ultralytics effdet torchmetrics pycocotools


## Configuration


In [ ]:

from pathlib import Path
import os
import random
import yaml
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DATA_YAML = Path('/content/tom_benh.v1i.yolov11/data.yaml')
OUTPUT_ROOT = Path('/content/drive/MyDrive/shrimp/runs_detection_models')

IMG_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 50
SEEDS = [0, 1, 2]
NUM_WORKERS = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('device:', DEVICE)
print('data yaml:', DATA_YAML)
print('output root:', OUTPUT_ROOT)


## Dataset Utilities


In [ ]:

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def load_data_yaml(path: Path):
    with open(path, 'r') as f:
        cfg = yaml.safe_load(f)
    root = Path(cfg.get('path', path.parent))
    if not root.is_absolute():
        root = (path.parent / root).resolve()

    names = cfg.get('names')
    if isinstance(names, dict):
        names = [names[i] for i in sorted(names)]

    split_dirs = {}
    for split in ['train', 'val', 'valid', 'test']:
        if split in cfg:
            split_path = Path(cfg[split])
            if not split_path.is_absolute():
                split_path = root / split_path
            split_dirs[split] = split_path

    if 'val' not in split_dirs and 'valid' in split_dirs:
        split_dirs['val'] = split_dirs['valid']

    return root, split_dirs, names


DATA_ROOT, SPLIT_DIRS, CLASS_NAMES = load_data_yaml(DATA_YAML)
NUM_CLASSES = len(CLASS_NAMES)
print('classes:', CLASS_NAMES)
print('splits:', SPLIT_DIRS)


class YoloDetectionDataset(Dataset):
    def __init__(self, images_dir: Path, img_size: int = 512):
        self.images_dir = Path(images_dir)
        self.img_size = img_size
        self.image_paths = sorted([
            p for p in self.images_dir.glob('*')
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
        ])
        if not self.image_paths:
            raise FileNotFoundError(f'No images found in {self.images_dir}')

    def __len__(self):
        return len(self.image_paths)

    def _label_path(self, image_path: Path) -> Path:
        parts = list(image_path.parts)
        if 'images' in parts:
            idx = parts.index('images')
            parts[idx] = 'labels'
            return Path(*parts).with_suffix('.txt')
        return image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        orig_w, orig_h = image.size
        image = image.resize((self.img_size, self.img_size))
        image_tensor = F.to_tensor(image)

        boxes = []
        labels = []
        label_path = self._label_path(image_path)
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                if not line.strip():
                    continue
                cls, xc, yc, w, h = map(float, line.split()[:5])
                x1 = (xc - w / 2) * self.img_size
                y1 = (yc - h / 2) * self.img_size
                x2 = (xc + w / 2) * self.img_size
                y2 = (yc + h / 2) * self.img_size
                x1 = max(0.0, min(float(self.img_size - 1), x1))
                y1 = max(0.0, min(float(self.img_size - 1), y1))
                x2 = max(0.0, min(float(self.img_size - 1), x2))
                y2 = max(0.0, min(float(self.img_size - 1), y2))
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)  # torchvision uses 0 as background

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx]),
            'area': torch.tensor([(b[2] - b[0]) * (b[3] - b[1]) for b in boxes], dtype=torch.float32),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image_tensor, target


def detection_collate(batch):
    return tuple(zip(*batch))


train_ds = YoloDetectionDataset(SPLIT_DIRS['train'], IMG_SIZE)
val_ds = YoloDetectionDataset(SPLIT_DIRS['val'], IMG_SIZE)
print('train images:', len(train_ds), 'val images:', len(val_ds))


## Faster R-CNN Training


In [ ]:

def build_faster_rcnn(num_classes: int):
    # num_classes includes background class.
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def train_faster_rcnn(seed: int):
    set_seed(seed)
    run_name = f'fasterrcnn_resnet50_fpn_seed{seed}'
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=detection_collate,
    )

    model = build_faster_rcnn(NUM_CLASSES + 1).to(DEVICE)
    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=0.005,
        momentum=0.9,
        weight_decay=0.0005,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

    history = []
    best_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for images, targets in train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        mean_loss = float(np.mean(losses))
        history.append({'epoch': epoch, 'train_loss': mean_loss})
        print(f'[{run_name}] epoch {epoch:03d}/{EPOCHS} train_loss={mean_loss:.4f}')

        if mean_loss < best_loss:
            best_loss = mean_loss
            torch.save({
                'model_state': model.state_dict(),
                'class_names': CLASS_NAMES,
                'img_size': IMG_SIZE,
                'epoch': epoch,
                'train_loss': mean_loss,
            }, run_dir / 'best.pt')

    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    torch.save(model.state_dict(), run_dir / 'last_state_dict.pt')
    return run_dir


for seed in SEEDS:
    train_faster_rcnn(seed)


## EfficientDet Training


In [ ]:

from effdet import create_model


class EfficientDetDataset(YoloDetectionDataset):
    def __getitem__(self, idx):
        image, target = super().__getitem__(idx)
        boxes_xyxy = target['boxes']
        if len(boxes_xyxy):
            # effdet expects yxyx boxes.
            boxes_yxyx = boxes_xyxy[:, [1, 0, 3, 2]]
            labels = target['labels'] - 1  # effdet classes are 0-based
        else:
            boxes_yxyx = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        eff_target = {
            'bbox': boxes_yxyx,
            'cls': labels,
            'img_scale': torch.tensor([1.0], dtype=torch.float32),
            'img_size': torch.tensor([self.img_size, self.img_size], dtype=torch.float32),
        }
        return image, eff_target


def efficientdet_collate(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    return images, list(targets)


eff_train_ds = EfficientDetDataset(SPLIT_DIRS['train'], IMG_SIZE)


def build_efficientdet(num_classes: int):
    return create_model(
        'tf_efficientdet_d0',
        bench_task='train',
        num_classes=num_classes,
        pretrained=True,
        image_size=(IMG_SIZE, IMG_SIZE),
    )


def train_efficientdet(seed: int):
    set_seed(seed)
    run_name = f'efficientdet_d0_seed{seed}'
    run_dir = OUTPUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    train_loader = DataLoader(
        eff_train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=efficientdet_collate,
    )

    model = build_efficientdet(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = []
    best_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for images, targets in train_loader:
            images = images.to(DEVICE)
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = loss_dict['loss'] if isinstance(loss_dict, dict) else loss_dict

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        mean_loss = float(np.mean(losses))
        history.append({'epoch': epoch, 'train_loss': mean_loss})
        print(f'[{run_name}] epoch {epoch:03d}/{EPOCHS} train_loss={mean_loss:.4f}')

        if mean_loss < best_loss:
            best_loss = mean_loss
            torch.save({
                'model_state': model.state_dict(),
                'class_names': CLASS_NAMES,
                'img_size': IMG_SIZE,
                'epoch': epoch,
                'train_loss': mean_loss,
            }, run_dir / 'best.pt')

    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    torch.save(model.state_dict(), run_dir / 'last_state_dict.pt')
    return run_dir


for seed in SEEDS:
    train_efficientdet(seed)


## RT-DETR Training


In [ ]:

from ultralytics import RTDETR


def train_rtdetr(seed: int):
    set_seed(seed)
    run_name = f'rtdetr_l_seed{seed}'
    print(f'========== TRAIN RT-DETR | seed={seed} | run={run_name} ==========')

    model = RTDETR('rtdetr-l.pt')
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=20,
        project=str(OUTPUT_ROOT),
        name=run_name,
        device=0 if DEVICE == 'cuda' else 'cpu',
        seed=seed,
        deterministic=True,
    )


for seed in SEEDS:
    train_rtdetr(seed)


## RT-DETR Test Evaluation


In [ ]:

from ultralytics import RTDETR

rows = []
for seed in SEEDS:
    run_name = f'rtdetr_l_seed{seed}'
    model_path = OUTPUT_ROOT / run_name / 'weights' / 'best.pt'
    if not model_path.exists():
        print('[WARN] missing:', model_path)
        continue

    model = RTDETR(str(model_path))
    metrics = model.val(
        data=str(DATA_YAML),
        split='test',
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=0 if DEVICE == 'cuda' else 'cpu',
        verbose=False,
    )
    rows.append({
        'model': 'RT-DETR-L',
        'seed': seed,
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'preprocess_ms': float(metrics.speed.get('preprocess', 0.0)),
        'inference_ms': float(metrics.speed.get('inference', 0.0)),
        'postprocess_ms': float(metrics.speed.get('postprocess', 0.0)),
    })

rtdetr_df = pd.DataFrame(rows)
rtdetr_df.to_csv(OUTPUT_ROOT / 'rtdetr_test_summary.csv', index=False)
rtdetr_df


## Notes

- Faster R-CNN checkpoint tốt nhất nằm ở `runs_detection_models/fasterrcnn_resnet50_fpn_seed*/best.pt`.
- EfficientDet checkpoint tốt nhất nằm ở `runs_detection_models/efficientdet_d0_seed*/best.pt`.
- RT-DETR dùng format output chuẩn của Ultralytics, checkpoint ở `runs_detection_models/rtdetr_l_seed*/weights/best.pt`.
- Nếu bị hết VRAM, giảm `BATCH_SIZE` xuống `2` hoặc `1`.
